# TT/MPS基礎 09 — Truncated SVD と Bond Rank Truncation

## 今回の位置づけ

前回の Notebook 08 では、3階 TT/MPS

$$
X
=
G_1^{[L]}
G_2^{[C]}
G_3^{[R]}
$$

を出発点として、第2中心コアの左展開

$$
A
=
G_2^{[C]\langle L\rangle}
\in
\mathbb{R}^{(r_1n_2)\times r_2}
$$

に exact SVD

$$
A
=
U\Sigma V^T
$$

を適用しました。

全ての SVD 成分を保持することで、

$$
X
=
G_1^{[L]}
G_2^{[L]}
\Sigma
\widetilde G_3^{[R]}
$$

という bond-centered な形を作り、切断

$$
(i_1,i_2)\mid i_3
$$

に対する Schmidt 形

$$
|X\rangle
=
\sum_{\beta=1}^{\rho}
\sigma_\beta
|L_\beta\rangle
\otimes
|R_\beta\rangle
$$

まで確認しました。

今回は、この Schmidt 成分を一部だけ残す **truncated SVD** を扱います。

### 今回やること

1. exact SVD と truncated SVD の違いを整理する
2. 小さい特異値を削除候補にできる理由を確認する
3. bond rank
   $$
   \rho\rightarrow k
   $$
   の意味とコア shape の変化を確認する
4. truncation 後に全テンソルが不変でなくなる理由を確認する
5. 単一 bond の truncation error
   $$
   \|X-\widetilde X_k\|_F^2
   =
   \sum_{\beta=k+1}^{\rho}\sigma_\beta^2
   $$
   を理論と数値の両方で確認する

### 今回はまだ扱わないもの

- Eckart–Young–Mirsky 定理の最適性の証明
- tolerance による rank 決定
- relative error による rank 決定
- TT rounding の sweep
- 複数 bond にまたがる truncation error
- entanglement entropy
- DMRG / ALS
- TT-matrix / MPO

この Notebook では、**3階 TT/MPS の第2–第3 bond を一度だけ truncation する場合**に限定します。


## 1. 出発点：Notebook 08 の Schmidt 形

切断

$$
(i_1,i_2)\mid i_3
$$

に対して、

$$
|X\rangle
=
\sum_{\beta=1}^{\rho}
\sigma_\beta
|L_\beta\rangle
\otimes
|R_\beta\rangle,
$$

$$
\sigma_1
\ge
\sigma_2
\ge
\cdots
\ge
\sigma_\rho
>
0
$$

を出発点にします。

ここで、

$$
\langle L_\beta|L_{\beta'}\rangle
=
\delta_{\beta\beta'},
\qquad
\langle R_\beta|R_{\beta'}\rangle
=
\delta_{\beta\beta'}
$$

なので、

$$
|L_\beta\rangle\otimes|R_\beta\rangle
$$

も互いに正規直交します。

したがって、

$$
\|X\|_F^2
=
\sum_{\beta=1}^{\rho}
\sigma_\beta^2.
$$

今回の truncation は、この直交展開の一部を削除する操作として理解します。


## 2. Exact SVD と Truncated SVD

第2中心コアの左展開を

$$
A
=
G_2^{[C]\langle L\rangle}
\in
\mathbb{R}^{(r_1n_2)\times r_2}
$$

とします。

rank を

$$
\rho
=
\operatorname{rank}(A)
$$

とすると、compact な exact SVD は

$$
A
=
U\Sigma V^T
=
\sum_{\beta=1}^{\rho}
\sigma_\beta
u_\beta v_\beta^T.
$$

全ての $\rho$ 成分を残す限り、これは完全な等式です。

一方、目標 rank を

$$
k<\rho
$$

として先頭の $k$ 成分だけを残すと、

$$
\widetilde A_k
=
U_k\Sigma_kV_k^T
=
\sum_{\beta=1}^{k}
\sigma_\beta
u_\beta v_\beta^T.
$$

shape は、

$$
U_k
\in
\mathbb{R}^{(r_1n_2)\times k},
$$

$$
\Sigma_k
\in
\mathbb{R}^{k\times k},
$$

$$
V_k^T
\in
\mathbb{R}^{k\times r_2}.
$$

exact SVD では

$$
A
=
\widetilde A_\rho,
$$

ですが、$k<\rho$ では非ゼロの SVD 成分を削除するため、

$$
\boxed{
\widetilde A_k\neq A
}
$$

です。

つまり、

$$
\boxed{
\text{exact SVD：表現を変えるだけ}
}
$$

に対して、

$$
\boxed{
\text{truncated SVD：低 rank への近似}
}
$$

となります。


## 3. 今回の数値検証用セットアップ

Notebook 08 と同じく、

$$
X
=
G_1^{[L]}
G_2^{[C]}
G_3^{[R]}
$$

という第2サイト中心の mixed-canonical form を出発点にします。

第1コアは左直交、第3コアは右直交になるよう、既習の QR で作ります。

一方、今回は truncated SVD の教材として、

> **小さい特異値を1本捨てる**

という状況を分かりやすくするため、中心コア $G_2^{[C]}$ は乱数 TT からそのまま作るのではなく、特異値を意図的に

$$
(\sigma_1,\sigma_2,\sigma_3)
=
(6,\;2,\;0.25)
$$

とした行列から構成します。

中心コアの左展開は、

$$
A
=
G_2^{[C]\langle L\rangle}
\in
\mathbb{R}^{(r_1n_2)\times r_2}
=
\mathbb{R}^{6\times3}.
$$

この例では、

$$
\rho=3,
\qquad
k=2,
\qquad
k<\rho
$$

です。

したがって $k=2$ の truncation では、第3 Schmidt 成分を削除します。

捨てるノルム二乗寄与は、

$$
\sigma_3^2
=
0.25^2
=
0.0625.
$$

全体のノルム二乗は、

$$
\|X\|_F^2
=
6^2+2^2+0.25^2
=
40.0625
$$

なので、捨てる成分は相対的には小さい例になっています。

以後は、

- `G1_left`
- `G2_center`
- `G3_right`
- `X_center2`

を出発点にします。

今回の truncation では、

$$
k=2
$$

を固定して使います。


In [1]:
import torch

torch.set_default_dtype(torch.float64)
torch.manual_seed(0)


def reconstruct_tt3(
    G1: torch.Tensor,
    G2: torch.Tensor,
    G3: torch.Tensor,
) -> torch.Tensor:
    """3個のTTコアから3階テンソルを再構成する。"""
    return torch.einsum("aib,bjc,ckd->ijk", G1, G2, G3)


# 小さい3階TTのshape
n1, n2, n3 = 4, 3, 5
r1, r2 = 2, 3

# --- 左右の直交コアを作る ---

# 第1コアを左直交化
G1_seed = torch.randn(1, n1, r1)
A1 = G1_seed.squeeze(0)
Q1, _ = torch.linalg.qr(A1, mode="reduced")
G1_left = Q1.unsqueeze(0)

# 第3コアを右直交化
G3_seed = torch.randn(r2, n3, 1)
A3 = G3_seed.squeeze(-1)
Q3, _ = torch.linalg.qr(A3.T, mode="reduced")
G3_right = Q3.T.unsqueeze(-1)

# --- 教材として明瞭な特異値減衰を持つ中心コアを構成する ---

# A = G2_center^{<L>} の shape は
# (r1 * n2, r2) = (6, 3)
A_random = torch.randn(r1 * n2, r2)

# reduced QR により、列直交な U_seed を作る
U_seed, _ = torch.linalg.qr(A_random, mode="reduced")

# 右特異ベクトル側の直交行列 V_seed を作る
V_seed, _ = torch.linalg.qr(
    torch.randn(r2, r2),
    mode="reduced",
)

# k=2 で第3成分を捨てたとき、
# 捨てる寄与が明瞭になるように特異値を設定する
S_target = torch.tensor([6.0, 2.0, 0.25])

# A = U Sigma V^T を明示的に構成する
A_center = U_seed @ torch.diag(S_target) @ V_seed.T

# (r1 * n2, r2) -> (r1, n2, r2) に戻す
G2_center = A_center.reshape(r1, n2, r2)

# この Notebook の基準となる mixed-canonical TT/MPS
X_center2 = reconstruct_tt3(
    G1_left,
    G2_center,
    G3_right,
)

print("G1_left  :", tuple(G1_left.shape))
print("G2_center:", tuple(G2_center.shape))
print("G3_right :", tuple(G3_right.shape))
print("target singular values:", S_target)


G1_left  : (1, 4, 2)
G2_center: (2, 3, 3)
G3_right : (3, 5, 1)
target singular values: tensor([6.0000, 2.0000, 0.2500])


## 4. 演習1 — Exact SVD から rank-$k$ の因子を取り出す

### 目的

まず、exact SVD と truncated SVD の違いを **SVD 成分を何本保持するか**という形で確認します。

中心コアを左展開して、

$$
A
=
G_2^{[C]\langle L\rangle}
$$

を作り、

$$
A
=
U\Sigma V^T
$$

とします。

その後、

$$
k=2
$$

として、

$$
U_k,\qquad
\Sigma_k,\qquad
V_k^T
$$

を取り出します。

`torch.linalg.svd` では、

```python
U, S, Vh = torch.linalg.svd(A, full_matrices=False)
```

として **reduced SVD** を明示します。

今回、

$$
A\in\mathbb{R}^{6\times3}
$$

なので、

$$
U\in\mathbb{R}^{6\times3},
\qquad
S\in\mathbb{R}^{3},
\qquad
V^T\in\mathbb{R}^{3\times3}.
$$

人工的に設定した3つの特異値は全て非ゼロなので、

$$
\rho=3.
$$

### TODO

1. `G2_center` を shape $(r_1n_2,r_2)$ に左展開する
2. reduced SVD を行う
3. `U`, `S`, `Vh` の shape を確認する
4. 保持する rank を `k = 2` とする
5. 先頭 $k$ 成分から `U_k`, `S_k`, `Vh_k` を作る
6. exact 側と truncated 側の shape の違いを確認する

### 考えること

- `S` の要素数と、数学上の $\rho$ は今回の入力でどう対応するか
- `U_k` の列数が $k$ になると、どの bond 次元が変わるか
- この段階ではまだ全テンソルを再構成していないこと


In [2]:
# TODO 1:
# Exact SVD から rank-k の因子を取り出してください。
#
from nn_compression.compression.svd import truncated_svd, matrix_max_rank

# 1. G2_center を左展開
A = G2_center.reshape(r1 * n2, r2)

# 2. rank と k の比較
# rank: 行列 A の数学的な最大 rank（= min(shape)、今回は exact SVD の本数）
# k   : 実際に残す truncated rank
rank = matrix_max_rank(A)
k = 2

print("A shape        :", tuple(A.shape))
print("rank (exact max):", rank)
print("k (keep)        :", k)
print("discarded       :", rank - k)
assert 1 <= k <= rank

# 3. reduced / exact SVD（特異値を全部取る）
U, S, Vh = truncated_svd(A, rank)

print("U shape :", tuple(U.shape))
print("S shape :", tuple(S.shape))
print("Vh shape:", tuple(Vh.shape))
print("singular values:", S)

# 4-5. 先頭 k 成分だけ残す
U_k = U[:, :k]
S_k = S[:k]
Vh_k = Vh[:k, :]

# 6. exact と truncated の shape を表示
print("exact     U,S,Vh :", tuple(U.shape), tuple(S.shape), tuple(Vh.shape))
print("truncated U,S,Vh :", tuple(U_k.shape), tuple(S_k.shape), tuple(Vh_k.shape))

A shape        : (6, 3)
rank (exact max): 3
k (keep)        : 2
discarded       : 1
U shape : (6, 3)
S shape : (3,)
Vh shape: (3, 3)
singular values: tensor([6.0000, 2.0000, 0.2500])
exact     U,S,Vh : (6, 3) (3,) (3, 3)
truncated U,S,Vh : (6, 2) (2,) (2, 3)


## 5. なぜ小さい特異値を削除候補にできるのか

Schmidt 形を

$$
|X\rangle
=
\sum_{\beta=1}^{\rho}
\sigma_\beta
|S_\beta\rangle,
$$

$$
|S_\beta\rangle
=
|L_\beta\rangle\otimes|R_\beta\rangle
$$

と書きます。

Schmidt 状態は正規直交しているので、

$$
\langle S_\beta|S_{\beta'}\rangle
=
\delta_{\beta\beta'}.
$$

したがって、

$$
\begin{aligned}
\|X\|_F^2
&=
\left\langle
\sum_{\beta=1}^{\rho}\sigma_\beta S_\beta
\middle|
\sum_{\beta'=1}^{\rho}\sigma_{\beta'}S_{\beta'}
\right\rangle
\\
&=
\sum_{\beta=1}^{\rho}
\sum_{\beta'=1}^{\rho}
\sigma_\beta\sigma_{\beta'}
\delta_{\beta\beta'}
\\
&=
\sum_{\beta=1}^{\rho}
\sigma_\beta^2.
\end{aligned}
$$

つまり、第 $\beta$ Schmidt 成分のノルム二乗への寄与は

$$
\boxed{
\sigma_\beta^2
}
$$

です。

特異値は

$$
\sigma_1
\ge
\sigma_2
\ge
\cdots
\ge
\sigma_\rho
$$

と並んでいるため、小さい特異値に対応する Schmidt 成分ほど、Frobenius ノルムの観点では小さい成分です。

### 「情報が小さい」の意味

ここで「情報が少ない」と断定するより、

> **Frobenius ノルムで測ったとき、その Schmidt 成分を削除して失う量が小さい**

と考えます。

削除されるのは単なるスカラー $\sigma_\beta$ ではなく、

$$
\sigma_\beta
|L_\beta\rangle
\otimes
|R_\beta\rangle
$$

という、左右を結ぶ一つの Schmidt 成分全体です。


## 6. 演習2 — 各特異値のノルム二乗への寄与を見る

### 目的

SVD で得た特異値について、

$$
\sigma_\beta^2
$$

を計算し、各 Schmidt 成分が全体のノルム二乗にどれだけ寄与しているかを確認します。

### TODO

1. `S` の各要素を二乗する
2. 全特異値の二乗和を求める
3. 先頭 $k$ 個の二乗和を求める
4. $k+1$ 番目以降の二乗和を求める

まだここでは、

$$
\|X-\widetilde X_k\|_F^2
$$

との一致確認までは行いません。

### 考えること

- `S[k:]` に入っている成分は何か
- 小さい特異値を落とすと、何を小さく抑えようとしているのか
- 「特異値だけをゼロにする」と「対応する rank-1 / Schmidt 成分を削除する」の違い


In [10]:
# TODO 2:
# 各特異値のノルム二乗への寄与を確認してください。
#
singular_value_energy = S**2
total_energy = singular_value_energy.sum() 
kept_energy = (torch.linalg.vector_norm(S_k))**2
discarded_energy = total_energy - kept_energy
#
# S と各エネルギーを表示してください。
print("S                 :", S.tolist())
print("sigma^2           :", singular_value_energy.tolist())
print("total_energy      :", total_energy.item())
print("kept_energy (k)   :", kept_energy.item() if torch.is_tensor(kept_energy) else kept_energy)
print("discarded_energy  :", discarded_energy.item() if torch.is_tensor(discarded_energy) else discarded_energy)
print("kept / total      :", (kept_energy / total_energy).item())

S                 : [6.0, 2.0000000000000004, 0.2500000000000006]
sigma^2           : [36.0, 4.000000000000002, 0.0625000000000003]
total_energy      : 40.0625
kept_energy (k)   : 40.00000000000001
discarded_energy  : 0.062499999999992895
kept / total      : 0.998439937597504


## 7. Bond rank $\rho\rightarrow k$ は何を意味するか

Notebook 08 では、SVD 後の bond 添字を

$$
\beta
=
1,\ldots,\rho
$$

とし、

$$
X
=
G_1^{[L]}
G_2^{[L]}
\Sigma
\widetilde G_3^{[R]}
$$

と書きました。

成分表示では、

$$
X(i_1,i_2,i_3)
=
\sum_{\beta=1}^{\rho}
L_\beta(i_1,i_2)
\sigma_\beta
R_\beta(i_3).
$$

truncation 後は、

$$
\widetilde X_k(i_1,i_2,i_3)
=
\sum_{\beta=1}^{k}
L_\beta(i_1,i_2)
\sigma_\beta
R_\beta(i_3).
$$

つまり、

$$
\boxed{
\beta=1,\ldots,\rho
\quad\longrightarrow\quad
\beta=1,\ldots,k
}
$$

です。

この bond に接続するコアの shape は、

$$
G_2^{[L]}
:
(r_1,n_2,\rho)
\longrightarrow
(r_1,n_2,k),
$$

$$
\widetilde G_3^{[R]}
:
(\rho,n_3,1)
\longrightarrow
(k,n_3,1).
$$

したがって bond rank の削減は、単なる配列サイズ変更ではありません。

> 切断の左右を結ぶ独立な Schmidt 成分・有効チャネルの本数を $\rho$ 本から $k$ 本へ減らす操作

です。


## 8. 演習3 — Truncated Core を作り、Bond Shape を確認する

### 目的

rank-$k$ truncated SVD を TT/MPS のコアへ戻したとき、

$$
\rho\rightarrow k
$$

が実際にコア shape にどう現れるかを確認します。

### TODO

1. `U_k` を reshape して truncated 第2コアを作る
2. `S_k` と `Vh_k` から右へ渡す因子を作る
3. その因子を `G3_right` の左 bond へ吸収して truncated 第3コアを作る
4. 新しい2つのコアの shape を表示する

期待する理論上の shape は、

$$
\widetilde G_2^{[L]}
\in
\mathbb{R}^{r_1\times n_2\times k},
$$

$$
\widetilde G_3^{[C]}
\in
\mathbb{R}^{k\times n_3\times1}.
$$

### 考えること

- どの添字の範囲が $\rho$ から $k$ に変化したか
- $n_1,n_2,n_3$ の物理次元は変化しているか
- truncation はどの bond にだけ入っているか


In [14]:
# TODO 3:
# rank-k の truncated core を作ってください。
#
# 1. U_k を第2コアへ戻す
G2_truncated = U_k.reshape(r1,n2,k)
#
# 2. Sigma_k V_k^T を作る
transfer_k = torch.diag(S_k)@Vh_k
#
# 3. G3_right に吸収して第3コアを作る
G3_truncated = torch.tensordot(transfer_k,G3_right,dims=([1],[0]))
#
# 4. shape を表示する
print("G2_truncated :", tuple(G2_truncated.shape))  # (r1, n2, k)
print("transfer_k   :", tuple(transfer_k.shape))    # (k, r2)
print("G3_truncated :", tuple(G3_truncated.shape))  # (k, n3, 1)

G2_truncated : (2, 3, 2)
transfer_k   : (2, 3)
G3_truncated : (2, 5, 1)


## 9. なぜ Truncation 後に $X$ は不変でないのか

exact SVD では、

$$
A
=
\sum_{\beta=1}^{\rho}
\sigma_\beta u_\beta v_\beta^T
$$

の全項を保持します。

一方、truncated SVD は、

$$
\widetilde A_k
=
\sum_{\beta=1}^{k}
\sigma_\beta u_\beta v_\beta^T
$$

です。

捨てた部分を

$$
E_k
=
\sum_{\beta=k+1}^{\rho}
\sigma_\beta u_\beta v_\beta^T
$$

と置けば、

$$
A
=
\widetilde A_k+E_k.
$$

$k<\rho$ なら、数学上の $\rho$ は非ゼロ特異値の本数なので、

$$
\sigma_{k+1}>0.
$$

したがって、

$$
E_k\neq0
$$

であり、

$$
\boxed{
\widetilde A_k\neq A
}
$$

です。

TT/MPS 全体でも、

$$
\boxed{
\widetilde X_k\neq X
}
$$

となります。

これは exact center move のような gauge transformation ではなく、**状態・テンソルそのものを低 bond rank の表現へ近似する操作**です。


## 10. 演習4 — Truncation 後に $X$ が変わることを確認する

### 目的

演習3で作った truncated core から、

$$
\widetilde X_k
$$

を再構成し、

$$
\widetilde X_k\neq X
$$

を数値で確認します。

### TODO

1. `G1_left`, truncated 第2コア, truncated 第3コアから `X_truncated` を再構成する
2. 
   $$
   \|X_{\mathrm{center2}}-X_{\mathrm{truncated}}\|_F
   $$
   を計算する
3. exact SVD center move の丸め誤差と違い、truncation error が明確に非ゼロになることを確認する

### 考えること

- 今回の非ゼロ誤差は浮動小数点丸めだけで説明できるか
- exact SVD のときと、何を実際に削除した点が違うか


In [16]:
# TODO 4:
# truncated TT/MPS を再構成し、元の X との差を確認してください。
#
X_truncated = reconstruct_tt3(G1_left,G2_truncated,G3_truncated)
#
truncation_error = torch.norm(X_center2 - X_truncated)
#
print("X_center2 shape   :", tuple(X_center2.shape))
print("X_truncated shape :", tuple(X_truncated.shape))
print("truncation error   :", truncation_error.item())

X_center2 shape   : (4, 3, 5)
X_truncated shape : (4, 3, 5)
truncation error   : 0.25000000000000006


## 11. なぜ誤差が「捨てた特異値の二乗和」になるのか

Schmidt 形は、

$$
|X\rangle
=
\sum_{\beta=1}^{\rho}
\sigma_\beta
|L_\beta\rangle\otimes|R_\beta\rangle.
$$

rank-$k$ truncation 後は、

$$
|\widetilde X_k\rangle
=
\sum_{\beta=1}^{k}
\sigma_\beta
|L_\beta\rangle\otimes|R_\beta\rangle.
$$

したがって差は、

$$
\begin{aligned}
|X\rangle-|\widetilde X_k\rangle
&=
\sum_{\beta=k+1}^{\rho}
\sigma_\beta
|L_\beta\rangle\otimes|R_\beta\rangle.
\end{aligned}
$$

ここで、

$$
|S_\beta\rangle
=
|L_\beta\rangle\otimes|R_\beta\rangle
$$

と書けば、

$$
\langle S_\beta|S_{\beta'}\rangle
=
\delta_{\beta\beta'}.
$$

よって、

$$
\begin{aligned}
\|X-\widetilde X_k\|_F^2
&=
\left\langle
\sum_{\beta=k+1}^{\rho}
\sigma_\beta S_\beta
\middle|
\sum_{\beta'=k+1}^{\rho}
\sigma_{\beta'}S_{\beta'}
\right\rangle
\\
&=
\sum_{\beta=k+1}^{\rho}
\sum_{\beta'=k+1}^{\rho}
\sigma_\beta
\sigma_{\beta'}
\delta_{\beta\beta'}
\\
&=
\boxed{
\sum_{\beta=k+1}^{\rho}
\sigma_\beta^2
}.
\end{aligned}
$$

したがって、

$$
\boxed{
\|X-\widetilde X_k\|_F
=
\sqrt{
\sum_{\beta=k+1}^{\rho}
\sigma_\beta^2
}
}
$$

です。

これは今回のように、**mixed-canonical / Schmidt 形になっている単一 bond を一度だけ truncation した場合**の誤差式です。


## 12. 残した成分と捨てた成分の直交分解

残した部分は、

$$
|\widetilde X_k\rangle
=
\sum_{\beta=1}^{k}
\sigma_\beta |S_\beta\rangle,
$$

捨てた部分は、

$$
|E_k\rangle
=
\sum_{\beta=k+1}^{\rho}
\sigma_\beta |S_\beta\rangle.
$$

両者は異なる Schmidt 成分からできているため直交します。

そのため、

$$
\boxed{
\|X\|_F^2
=
\|\widetilde X_k\|_F^2
+
\|X-\widetilde X_k\|_F^2
}
$$

が成り立ちます。

各項は、

$$
\|\widetilde X_k\|_F^2
=
\sum_{\beta=1}^{k}\sigma_\beta^2,
$$

$$
\|X-\widetilde X_k\|_F^2
=
\sum_{\beta=k+1}^{\rho}\sigma_\beta^2.
$$

つまり、元のノルム二乗が

$$
\text{残した Schmidt 成分}
+
\text{捨てた Schmidt 成分}
$$

へ直交分解されています。


## 13. 演習5 — 実測 Truncation Error と捨てた特異値を比較する

### 目的

理論式

$$
\|X-\widetilde X_k\|_F^2
=
\sum_{\beta=k+1}^{\rho}
\sigma_\beta^2
$$

を数値で確認します。

### TODO

次の量をそれぞれ計算してください。

1. 実際の tensor error
   $$
   \|X_{\mathrm{center2}}-X_{\mathrm{truncated}}\|_F^2
   $$
2. 捨てた特異値の二乗和
   $$
   \sum_{\beta=k+1}^{\rho}\sigma_\beta^2
   $$
3. その2つの差
4. 元テンソルのノルム二乗
5. truncated tensor のノルム二乗
6. 
   $$
   \|X\|_F^2
   -
   \left(
   \|\widetilde X_k\|_F^2
   +
   \|X-\widetilde X_k\|_F^2
   \right)
   $$
   が丸め誤差水準になること

### 考えること

- なぜ tensor error と discarded singular-value energy が一致するのか
- なぜ交差項が残らないのか
- この等式を複数 bond の truncation にそのまま適用してよいか


In [17]:
# TODO 5:
# 理論上の discarded weight と実測 truncation error を比較してください。
#
from torch.linalg import vector_norm

# 1. ||X - X_truncated||_F^2
tensor_error_sq = vector_norm(X_center2 - X_truncated).square().item()
#
# 2. sum_{beta>k} sigma_beta^2
discarded_singular_value_sq = S[k:].square().sum().item()
#
# 3. 両者の差
error_identity_gap = abs(tensor_error_sq - discarded_singular_value_sq)
#
# 4. Pythagoras 型のノルム分解
X_norm_sq = vector_norm(X_center2).square().item()
X_truncated_norm_sq = vector_norm(X_truncated).square().item()
pythagoras_gap = abs(
    X_norm_sq - (X_truncated_norm_sq + tensor_error_sq)
)

print("||X - X_truncated||_F^2 =", tensor_error_sq)
print("sum_{beta>k} sigma^2   =", discarded_singular_value_sq)
print("error identity gap     =", error_identity_gap)
print("||X||_F^2              =", X_norm_sq)
print("||X_truncated||_F^2    =", X_truncated_norm_sq)
print("Pythagoras gap         =", pythagoras_gap)

||X - X_truncated||_F^2 = 0.06250000000000003
sum_{beta>k} sigma^2   = 0.0625000000000003
error identity gap     = 2.7755575615628914e-16
||X||_F^2              = 40.062500000000014
||X_truncated||_F^2    = 39.999999999999986
Pythagoras gap         = 2.842170943040401e-14


## 14. 「何を捨てたのか」を物理・MPSの言葉で整理する

truncation で捨てるのは、単なる配列の末尾や単なる数値ではありません。

Schmidt 形

$$
|X\rangle
=
\sum_{\beta=1}^{\rho}
\sigma_\beta
|L_\beta\rangle\otimes|R_\beta\rangle
$$

における、

$$
\sigma_\beta
|L_\beta\rangle\otimes|R_\beta\rangle
$$

という **左右の対応した Schmidt 成分**です。

rank を

$$
\rho\rightarrow k
$$

にすると、切断をまたいで保持する有効状態ペアを $k$ 個に制限します。

量子状態として見る場合、Schmidt 状態は縮約密度行列の固有状態としても現れます。小さい Schmidt 係数に対応する状態を捨てることは、環境との結び付きが弱い有効状態を基底から除外することに対応します。

一方、一般の数値テンソル圧縮として TT/MPS を使う場合には、

$$
\sigma_\beta^2
$$

を確率と解釈する必要はありません。

その場合も、

> 第 $\beta$ Schmidt 成分がテンソル全体の Frobenius ノルム二乗にどれだけ寄与するか

という意味は変わりません。


## 15. 今回のまとめ

### Exact SVD

$$
A
=
\sum_{\beta=1}^{\rho}
\sigma_\beta u_\beta v_\beta^T
$$

の全成分を保持します。

$$
\boxed{
X_{\mathrm{after}}
=
X_{\mathrm{before}}
}
$$

であり、近似は入りません。

### Truncated SVD

$$
\widetilde A_k
=
\sum_{\beta=1}^{k}
\sigma_\beta u_\beta v_\beta^T,
\qquad
k<\rho
$$

とし、後半の Schmidt 成分を削除します。

そのため、

$$
\boxed{
\widetilde X_k\neq X
}
$$

です。

### Bond rank

$$
\boxed{
\rho\rightarrow k
}
$$

とは、切断をまたぐ独立な Schmidt 成分・有効チャネルの本数を $k$ に制限することです。

### Truncation error

$$
\boxed{
\|X-\widetilde X_k\|_F^2
=
\sum_{\beta=k+1}^{\rho}
\sigma_\beta^2
}
$$

です。

今回確認するのは、**単一 bond の rank truncation** までです。

次の段階では、

- なぜ先頭 $k$ 個を残す truncated SVD が最良の rank-$k$ 近似になるのか
- Eckart–Young–Mirsky 定理
- rank $k$ をどう選ぶか

へ進めます。

TT rounding や複数 bond の誤差評価は、さらにその後の段階です。
